In [ ]:
import json
from dotenv import load_dotenv
from pathlib import Path


# echo "GIFT_EVAL=PATH_TO_SAVE" >> .env
env_file = Path(".env")
key = "GIFT_EVAL"
value = "/home/niuyiming/gift-eval/downloaded_data"

if env_file.exists():
    lines = env_file.read_text().splitlines()
    for i, line in enumerate(lines):
        if line.startswith(f"{key}="):
            lines[i] = f"{key}={value}"
            break
    else:
        lines.append(f"{key}={value}")
    env_file.write_text("\n".join(lines) + "\n")
else:
    env_file.write_text(f"{key}={value}\n")

# Load environment variables
load_dotenv()

# short_datasets = "m4_yearly m4_quarterly m4_monthly m4_weekly m4_daily m4_hourly electricity/15T electricity/H electricity/D electricity/W solar/10T solar/H solar/D solar/W hospital covid_deaths us_births/D us_births/M us_births/W saugeenday/D saugeenday/M saugeenday/W temperature_rain_with_missing kdd_cup_2018_with_missing/H kdd_cup_2018_with_missing/D car_parts_with_missing restaurant hierarchical_sales/D hierarchical_sales/W LOOP_SEATTLE/5T LOOP_SEATTLE/H LOOP_SEATTLE/D SZ_TAXI/15T SZ_TAXI/H M_DENSE/H M_DENSE/D ett1/15T ett1/H ett1/D ett1/W ett2/15T ett2/H ett2/D ett2/W jena_weather/10T jena_weather/H jena_weather/D"
short_datasets = "bitbrains_fast_storage/5T bitbrains_fast_storage/H bitbrains_rnd/5T bitbrains_rnd/H bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H"

# med_long_datasets
med_long_datasets = ""

# Get union of short and med_long datasets
all_datasets = list(set(short_datasets.split() + med_long_datasets.split()))

dataset_properties_map = json.load(open("dataset_properties.json"))

In [5]:
from gluonts.ev.metrics import (
    MSE,
    MAE,
    MASE,
    MAPE,
    SMAPE,
    MSIS,
    RMSE,
    NRMSE,
    ND,
    MeanWeightedSumQuantileLoss,
)

# Instantiate the metrics
metrics = [
    MSE(forecast_type="mean"),
    MSE(forecast_type=0.5),
    MAE(),
    MASE(),
    MAPE(),
    SMAPE(),
    MSIS(),
    RMSE(),
    NRMSE(),
    ND(),
    MeanWeightedSumQuantileLoss(
        quantile_levels=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
    ),
]

In [ ]:
from gluonts.model import evaluate_model
from PhaseFormer.PhaseFormerEstimator import PhaseFormerEstimator
import csv
import os
import time
from gluonts.time_feature import get_seasonality
from gift_eval.data import Dataset

# Iterate over all available datasets

output_dir = "../results/phaseformer_evaluation"
# Ensure the output directory exists
os.makedirs(output_dir, exist_ok=True)

# Define the path for the CSV file
csv_file_path = os.path.join(output_dir, "all_results.csv")

pretty_names = {
    "saugeenday": "saugeen",
    "temperature_rain_with_missing": "temperature_rain",
    "kdd_cup_2018_with_missing": "kdd_cup_2018",
    "car_parts_with_missing": "car_parts",
}

with open(csv_file_path, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)

    # Write the header
    writer.writerow(
        [
            "dataset",
            "model",
            "eval_metrics/MSE[mean]",
            "eval_metrics/MSE[0.5]",
            "eval_metrics/MAE[0.5]",
            "eval_metrics/MASE[0.5]",
            "eval_metrics/MAPE[0.5]",
            "eval_metrics/sMAPE[0.5]",
            "eval_metrics/MSIS",
            "eval_metrics/RMSE[mean]",
            "eval_metrics/NRMSE[mean]",
            "eval_metrics/ND[0.5]",
            "eval_metrics/mean_weighted_sum_quantile_loss",
            "domain",
            "num_variates",
        ]
    )

for ds_name in all_datasets:
    print(f"Processing dataset: {ds_name}")
    terms = ["short", "medium", "long"]
    for term in terms:
        if (term == "medium" or term == "long") and ds_name not in med_long_datasets.split():
            continue

        # 解析 dataset key / freq
        if "/" in ds_name:
            ds_key = ds_name.split("/")[0]
            ds_freq = ds_name.split("/")[1]
            ds_key = ds_key.lower()
            ds_key = pretty_names.get(ds_key, ds_key)
        else:
            ds_key = ds_name.lower()
            ds_key = pretty_names.get(ds_key, ds_key)
            ds_freq = dataset_properties_map[ds_key]["frequency"]

        ds_config = f"{ds_key}/{ds_freq}/{term}"

        # 初始化 dataset，保留 multivariate 信息
        to_univariate = (
            False
            if Dataset(name=ds_name, term=term, to_univariate=False).target_dim == 1
            else True
        )
        dataset = Dataset(name=ds_name, term=term, to_univariate=to_univariate)

        # 计算 seasonality
        season_length = get_seasonality(dataset.freq)
        pred_len = dataset.prediction_length

        # 推断数据可用的最小上下文（和 TTM 一样）
        all_lengths = []
        num_channels = 1
        for x in dataset.test_data:
            tgt = x[0]["target"]
            if len(tgt.shape) == 1:
                all_lengths.append(len(tgt))
                num_channels = 1
            else:
                # 如果 target 的 shape 是 (C, L) 或 (L, C) 的话，按你的 Dataset 格式调整
                # 这里假设 target shape 为 (C, L) 或 (L, C) 中的一种；若与你的数据不同请调整
                if tgt.shape[0] == dataset.target_dim:
                    # (C, L) 风格
                    all_lengths.append(tgt.shape[1])
                    num_channels = tgt.shape[0]
                else:
                    # (L, C) 风格
                    all_lengths.append(tgt.shape[0])
                    num_channels = tgt.shape[1]

        if len(all_lengths) == 0:
            # 兜底值（不应该发生）
            min_context_length = max(season_length * 4, 720)
        else:
            min_context_length = min(all_lengths)

        print("Minimum context length among all time series in this dataset =", min_context_length)
        print("Number of channels detected =", num_channels)
        print("Season length detected =", season_length)

        # seq_len 依然按 seasonality 的 heuristic 计算（与原 notebook 保持一致）
        seq_len = max(season_length * 4, 720)
        context_length = min(min_context_length, seq_len)

        # prediction channels
        prediction_channel_indices = list(range(num_channels))
        if dataset.prediction_length > globals().get("TTM_MAX_FORECAST_HORIZON", pred_len):
            prediction_channel_indices = list(range(num_channels))

        seq_len = context_length = 720
        period_len = 8
        
        print("prediction_channel_indices =", prediction_channel_indices)
        print(f"Number of channels in the dataset {ds_name} =", num_channels)
        print(f"Using seq_len={seq_len}, context_length={context_length}, pred_len={pred_len}, perid_len={period_len}")

        batch_size = 32
        lr = 1e-3
        num_epochs = 30
        print(f"Training PhaseFormer on lr{lr} and bsz{batch_size} for {num_epochs} epochs...")

        # 导入 lightning 回调以启用 EarlyStopping（与原 notebook 保持兼容）
        import lightning.pytorch as pl

        # 实例化 PhaseFormerEstimator（传入数据驱动的 context/seq_len 与常用训练超参）
        estimator = PhaseFormerEstimator(
            lr=lr,
            prediction_length=pred_len,
            context_length=context_length,
            seq_len=seq_len,
            period_len=period_len,
            enc_in=dataset.target_dim if dataset.target_dim > 1 else 1,
            batch_size=batch_size if batch_size is not None else 32,
            trainer_kwargs=dict(
                max_epochs=20,
                accelerator="gpu",
                devices=[0],
                callbacks=[pl.callbacks.EarlyStopping(monitor="val_loss", patience=5, mode="min")],
            ),
            phase_attn_heads=1,
            distr_output=None,
        )

        predictor = estimator.train(training_data=dataset.training_dataset, validation_data=dataset.validation_dataset)

        # Evaluate
        res = evaluate_model(
            predictor,
            test_data=dataset.test_data,
            metrics=metrics,
            batch_size=512,
            axis=None,
            mask_invalid_label=True,
            allow_nan_forecast=False,
            seasonality=season_length,
        )

        # Append results
        with open(csv_file_path, "a", newline="") as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(
                [
                    ds_config,
                    "phaseformer",
                    res["MSE[mean]"][0],
                    res["MSE[0.5]"][0],
                    res["MAE[0.5]"][0],
                    res["MASE[0.5]"][0],
                    res["MAPE[0.5]"][0],
                    res["sMAPE[0.5]"][0],
                    res["MSIS"][0],
                    res["RMSE[mean]"][0],
                    res["NRMSE[mean]"][0],
                    res["ND[0.5]"][0],
                    res["mean_weighted_sum_quantile_loss"][0],
                    dataset_properties_map[ds_key]["domain"],
                    dataset_properties_map[ds_key]["num_variates"],
                ]
            )

        print(f"Results for {ds_name} have been written to {csv_file_path}")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]

  | Name  | Type  | Params | Mode 
----------------------------------------
0 | model | Model | 2.0 K  | train
----------------------------------------
2.0 K     Trainable params
0         Non-trainable params
2.0 K     Total params
0.008     Total estimated model params size (MB)
32        Modules in train mode
0         Modules in eval mode


Processing dataset: bizitobs_application
Minimum context length among all time series in this dataset = 7934
Number of channels detected = 1
Season length detected = 360
prediction_channel_indices = [0]
Number of channels in the dataset bizitobs_application = 1
Using seq_len=720, context_length=720, pred_len=60, perid_len=8
Training PhaseFormer on lr0.001 and bsz32 for 30 epochs...
[PhaseFormer] period_len=8 adjustment: seq_len 720 -> used_input_len 720; pred_len 60 -> used_output_len 64
Epoch 0: |          | 50/? [00:01<00:00, 28.88it/s, v_num=112, train_loss_step=8.41e+6, val_loss=4.92e+6, train_loss_epoch=4.23e+7]

Epoch 0, global step 50: 'val_loss' reached 4920797.50000 (best 4920797.50000), saving model to '/home/niuyiming/gift-eval/notebooks/lightning_logs/version_112/checkpoints/epoch=0-step=50.ckpt' as top 1


Epoch 1: |          | 50/? [00:01<00:00, 28.82it/s, v_num=112, train_loss_step=6.81e+6, val_loss=2.11e+6, train_loss_epoch=3e+7]   

Epoch 1, global step 100: 'val_loss' reached 2113709.25000 (best 2113709.25000), saving model to '/home/niuyiming/gift-eval/notebooks/lightning_logs/version_112/checkpoints/epoch=1-step=100.ckpt' as top 1


...

Epoch 10: |          | 50/? [00:02<00:00, 18.48it/s, v_num=119, train_loss_step=8.57e+6, val_loss=9.79e+6, train_loss_epoch=4.41e+6]

Epoch 10, global step 550: 'val_loss' was not in top 1


Epoch 10: |          | 50/? [00:02<00:00, 18.45it/s, v_num=119, train_loss_step=8.57e+6, val_loss=9.79e+6, train_loss_epoch=4.41e+6]


5000it [00:07, 680.74it/s]

Results for bitbrains_fast_storage/H have been written to ../results/phaseformer_evaluation/all_results.csv
